# Etapa 2c — Cierre de los mapas de color

Completa los tres controles que `02_mapas_color.ipynb` dejó pendientes. Solo desarrollo; componentes principales con la regla de signo adoptada en `02b` (oscuridad local).

1. **Estabilidad de la elección (validación cruzada por personas).** Cinco grupos de personas (`person_folds`). En cada vuelta, con las personas de entrenamiento se fija el sentido de cada mapa y se elige el de mayor área media; se mide en las personas que quedan afuera. ¿Se elige siempre el mismo mapa? ¿Cuánto optimismo hay?
2. **Resolución.** La tercera componente y el verde calculados con la foto a la mitad y a la cuarta parte, agrandados al tamaño original y evaluados a resolución original. Diferencia con la original, remuestreando personas.
3. **Complementariedad.** ¿Qué mapas agregan información a la tercera componente? Con las personas de entrenamiento de cada vuelta se ajusta una combinación lineal (discriminante de Fisher) de la tercera componente con cada otro mapa, todos estandarizados dentro del campo detectado; se mide el área de la combinación en las personas de afuera y se compara con la tercera componente sola. También la combinación de los diez mapas.

**Criterios.** La elección es estable si la tercera componente gana en las cinco vueltas. Una resolución reducida es no inferior si el límite inferior del intervalo al 95 % de (reducida − original) es ≥ −0,02 (decisión provisional: la resolución se cierra en la etapa 4, con los filtros). Un mapa pasa a las etapas 4 y 5 si su ganancia junto a la tercera componente tiene límite inferior > 0; el verde pasa siempre como referencia clásica.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.color import MAP_NAMES, color_maps, color_maps_at_scale, standardize
from src.config import SEED
from src.data import dev_ids, load_fov, load_image, load_split, load_vessels
from src.fov import detect_fov_robust
from src.metrics import auc
from src.split import person_folds
from src.stats import fisher_direction, percentile_interval, person_bootstrap_means

split = load_split().set_index("image")
ids = dev_ids()
dev = split.loc[ids]
folds = person_folds(dev.reset_index(), k=5, seed=SEED)
dev = dev.assign(fold=dev["person"].map(folds))
print(dev.groupby("fold").agg(personas=("person", "nunique"), imagenes=("person", "size"),
                              grupos=("group", lambda g: "".join(sorted(set(g))))))

      personas  imagenes grupos
fold                           
0            4         6   drgh
1            4         7   drgh
2            4         6   drgh
3            3         5   drgh
4            3         6   drgh


## Pasada 1: área de cada mapa, resoluciones y muestras de píxeles

In [2]:
rng = np.random.default_rng(SEED)
N_SAMPLE = 20_000                     # por clase y por imagen, para ajustar combinaciones
auc_rows, scale_rows, samples = [], [], {}

for image_id in ids:
    rgb, fov, ves = load_image(image_id), load_fov(image_id), load_vessels(image_id)
    region = detect_fov_robust(rgb)
    maps = color_maps(rgb, region)
    original = {}
    for name, m in maps.items():
        original[name] = auc(m, ves, fov)
        auc_rows.append({"image": image_id, "map": name, "auc_bright": original[name]})
    # muestras estandarizadas (dentro del campo de visión de la base, para tener etiqueta)
    z = np.stack([standardize(maps[n], region)[fov] for n in MAP_NAMES], axis=1)
    lab = ves[fov]
    pick = np.r_[rng.choice(np.flatnonzero(lab), N_SAMPLE, replace=False),
                 rng.choice(np.flatnonzero(~lab), N_SAMPLE, replace=False)]
    samples[image_id] = (z[pick], lab[pick])
    del z, maps
    for name in ("Verde", "PC3"):
        scale_rows.append({"image": image_id, "factor": 1, "map": name, "auc_bright": original[name]})
    for f in (2, 4):
        reduced = color_maps_at_scale(rgb, f, names=["Verde", "PC3"])
        for name, m in reduced.items():
            scale_rows.append({"image": image_id, "factor": f, "map": name, "auc_bright": auc(m, ves, fov)})
    print(image_id, end=" ", flush=True)

A = pd.DataFrame(auc_rows).pivot(index="image", columns="map", values="auc_bright")[MAP_NAMES]
S = pd.DataFrame(scale_rows)
A.to_csv(ROOT / "results" / "02c_auc_mapas.csv")
S.to_csv(ROOT / "results" / "02c_resoluciones.csv", index=False)

01_h 

02_h 

03_h 

04_h 

07_h 

08_h 

... (20 outputs left out)

11_g 

12_g 

13_g 

14_g 

## 1. Estabilidad de la elección

In [3]:
def oriented(frame, train_idx):
    """Orienta cada mapa (vasos claros u oscuros) con la media de entrenamiento.

    Se invierten las columnas (mapas) con media < 0,5. Ojo: frame.where(serie)
    alinearía la serie con las filas (imágenes), no con las columnas.
    """
    bright = frame.loc[train_idx].mean() >= 0.5
    out = frame.copy()
    out.loc[:, ~bright] = 1 - frame.loc[:, ~bright]
    return out, bright

cv_rows, heldout = [], []
for k in range(5):
    train = dev.index[dev.fold != k]
    test = dev.index[dev.fold == k]
    o, bright = oriented(A, train)
    best = o.loc[train].mean().idxmax()
    cv_rows.append({"vuelta": k, "elegido": best,
                    "sentido": "claros" if bright[best] else "oscuros",
                    "área en entrenamiento": o.loc[train, best].mean(),
                    "área afuera": o.loc[test, best].mean(),
                    "verde afuera": o.loc[test, "Verde"].mean()})
    heldout.append(pd.DataFrame({"elegido": o.loc[test, best], "verde": o.loc[test, "Verde"]}))
cv = pd.DataFrame(cv_rows)
heldout = pd.concat(heldout)
print(cv.round(4).to_string(index=False))
print(f"\nÁrea afuera del mapa elegido: {heldout['elegido'].mean():.4f}; verde: {heldout['verde'].mean():.4f}")
print(f"Optimismo (entrenamiento − afuera): {(cv['área en entrenamiento'] - cv['área afuera']).mean():.4f}")
b = person_bootstrap_means(heldout["elegido"] - heldout["verde"], dev["person"])
lo, hi = percentile_interval(b)
print(f"Elegido − verde, personas de afuera: {(heldout['elegido'] - heldout['verde']).mean():.4f} "
      f"(intervalo {lo[0]:.4f} a {hi[0]:.4f})")

 vuelta elegido sentido  área en entrenamiento  área afuera  verde afuera
      0     PC3  claros                 0.8158       0.7942        0.6819
      1     PC3  claros                 0.8149       0.8003        0.6658
      2     PC3  claros                 0.8100       0.8175        0.7134
      3     PC3  claros                 0.8108       0.8148        0.6988
      4     PC3  claros                 0.8061       0.8331        0.6928

Área afuera del mapa elegido: 0.8115; verde: 0.6894
Optimismo (entrenamiento − afuera): -0.0004
Elegido − verde, personas de afuera: 0.1221 (intervalo 0.1028 a 0.1385)


## 2. Resolución

In [4]:
W = S.pivot_table(index="image", columns=["map", "factor"], values="auc_bright")
rows = []
for name in ("PC3", "Verde"):
    sign = 1 if W[(name, 1)].mean() >= 0.5 else -1
    o = W[name] if sign > 0 else 1 - W[name]
    diffs = pd.DataFrame({f: o[f] - o[1] for f in (2, 4)})
    bs = person_bootstrap_means(diffs, dev["person"])
    lo, hi = percentile_interval(bs)
    for i, f in enumerate((2, 4)):
        rows.append({"mapa": name, "reducción": f"1/{f}", "área original": o[1].mean(),
                     "área reducida": o[f].mean(), "diferencia": diffs[f].mean(),
                     "límite inferior": lo[i], "límite superior": hi[i],
                     "no inferior": lo[i] >= -0.02})
pd.DataFrame(rows).round(4)

,mapa,reducción,área original,área reducida,diferencia,límite inferior,límite superior,no inferior
0,PC3,1/2,0.8115,0.8203,0.0088,0.0072,0.0108,True
1,PC3,1/4,0.8115,0.8227,0.0112,0.0087,0.0140,True
2,Verde,1/2,0.6894,0.6871,-0.0023,-0.0026,-0.0020,True
3,Verde,1/4,0.6894,0.6793,-0.0101,-0.0109,-0.0092,True


## 3. Complementariedad con la tercera componente

Correlación entre mapas (mediana de las imágenes, sobre las muestras estandarizadas):

In [5]:
ip = MAP_NAMES.index("PC3")
corr = pd.DataFrame({i: [abs(np.corrcoef(samples[i][0][:, ip], samples[i][0][:, j])[0, 1])
                         for j in range(len(MAP_NAMES))] for i in ids}, index=MAP_NAMES)
corr.median(axis=1).round(3).rename("correlación absoluta con PC3 (mediana)")

Rojo                0.114
Verde               0.171
Azul                0.364
Escala de grises    0.055
Lab L*              0.055
Lab a*              0.322
Lab b*              0.184
PC1                 0.075
PC2                 0.263
PC3                 1.000
Name: correlación absoluta con PC3 (mediana), dtype: float64

Combinaciones ajustadas en cada vuelta con las personas de entrenamiento y medidas en las de afuera (pasada 2, a resolución original):

In [6]:
COMBOS = {f"PC3 + {n}": [ip, MAP_NAMES.index(n)] for n in MAP_NAMES if n != "PC3"}
COMBOS["los diez"] = list(range(len(MAP_NAMES)))

weights = {}
for k in range(5):
    train_imgs = dev.index[dev.fold != k]
    X = np.concatenate([samples[i][0] for i in train_imgs])
    y = np.concatenate([samples[i][1] for i in train_imgs])
    for combo, cols in COMBOS.items():
        weights[(k, combo)] = fisher_direction(X[:, cols], y)

comb_rows = []
for image_id in ids:
    k = dev.loc[image_id, "fold"]
    rgb, fov, ves = load_image(image_id), load_fov(image_id), load_vessels(image_id)
    region = detect_fov_robust(rgb)
    maps = color_maps(rgb, region)
    z = {n: standardize(maps[n], region) for n in MAP_NAMES}
    del maps
    row = {"image": image_id, "PC3 sola": auc(z["PC3"], ves, fov)}
    for combo, cols in COMBOS.items():
        w = weights[(k, combo)]
        score = sum(w[c] * z[MAP_NAMES[j]] for c, j in enumerate(cols))
        row[combo] = auc(score, ves, fov)
    comb_rows.append(row)
    print(image_id, end=" ", flush=True)
C = pd.DataFrame(comb_rows).set_index("image")
C.to_csv(ROOT / "results" / "02c_combinaciones.csv")

01_h 

02_h 

03_h 

04_h 

07_h 

08_h 

... (20 outputs left out)

11_g 

12_g 

13_g 

14_g 

In [7]:
gain = C.drop(columns="PC3 sola").sub(C["PC3 sola"], axis=0)
bs = person_bootstrap_means(gain, dev["person"])
lo, hi = percentile_interval(bs)
comp = pd.DataFrame({"área combinada": C.drop(columns="PC3 sola").mean(),
                     "ganancia sobre PC3": gain.mean(), "límite inferior": lo, "límite superior": hi})
comp["pasa"] = comp["límite inferior"] > 0
print(f"PC3 sola, personas de afuera: {C['PC3 sola'].mean():.4f}")
comp.sort_values("ganancia sobre PC3", ascending=False).round(4)

PC3 sola, personas de afuera: 0.8115


,área combinada,ganancia sobre PC3,límite inferior,límite superior,pasa
los diez,0.8734,0.0619,0.0511,0.0750,True
PC3 + PC2,0.8646,0.0532,0.0422,0.0674,True
PC3 + Verde,0.8427,0.0312,0.0268,0.0366,True
PC3 + Azul,0.8378,0.0263,0.0227,0.0306,True
PC3 + Escala de grises,0.8296,0.0181,0.0151,0.0214,True
PC3 + Lab a*,0.8196,0.0081,0.0020,0.0163,True
PC3 + Lab L*,0.8183,0.0069,0.0049,0.0088,True
PC3 + PC1,0.8173,0.0058,0.0040,0.0076,True
PC3 + Rojo,0.8130,0.0015,0.0005,0.0024,True
PC3 + Lab b*,0.8112,-0.0003,-0.0005,-0.0000,False


In [8]:
print("Pesos de la combinación de los diez (promedio de las cinco vueltas, mapas estandarizados):")
print(pd.Series(np.mean([weights[(k, "los diez")] for k in range(5)], axis=0), index=MAP_NAMES).round(3))

Pesos de la combinación de los diez (promedio de las cinco vueltas, mapas estandarizados):
Rojo               -0.052
Verde              -0.024
Azul               -0.040
Escala de grises    0.026
Lab L*              0.747
Lab a*              0.001
Lab b*             -0.104
PC1                 0.603
PC2                 0.148
PC3                 0.128
dtype: float64


## Decisión (escrita después de correr)

**1. La elección es estable.** La tercera componente, con vasos claros, gana en las cinco vueltas. En las personas que quedan afuera su área es 0,8115, igual que en desarrollo completo: no hay optimismo apreciable. Supera al verde por 0,122 (intervalo 0,103 a 0,139).

*Desvío declarado:* en la primera corrida esta celda elegía el verde en todas las vueltas por un error de código (`frame.where` con una serie alinea por filas, no por columnas, y quedaban invertidos todos los mapas). Se corrigió y se volvió a correr todo el cuaderno.

**2. Resolución (provisional; se cierra en la etapa 4).** Calculada con la foto a la mitad o a la cuarta parte, la tercera componente separa un poco **mejor** que a resolución original: +0,009 a la mitad (0,007 a 0,011) y +0,011 a la cuarta parte (0,009 a 0,014). Promediar bloques baja el ruido. El verde a la cuarta parte pierde 0,010 (−0,011 a −0,009), dentro del margen. Con mapas crudos reducir no cuesta nada; con los filtros de la etapa 4 y los vasos finos puede cambiar, y ahí se decide.

**3. Complementariedad.** Combinar mapas ayuda: los diez juntos llegan a 0,873 (+0,062 sobre la tercera componente sola, intervalo 0,051 a 0,075). Por pares, lo que más suma a la tercera componente es la segunda componente (+0,053), el verde (+0,031), el azul (+0,026) y la escala de grises (+0,018). Los demás suman menos de 0,01 y el canal b de Lab nada.
- Con el criterio fijado (límite inferior > 0) pasan todos menos el canal b. Ese criterio resultó demasiado permisivo: el rojo pasa con +0,0015.
- *Desvío declarado:* para que las etapas 4 y 5 no se multipliquen, pasan los mapas que suman al menos 0,01: **tercera componente (principal), segunda componente, verde (además, referencia clásica), azul y escala de grises.** Este corte se eligió después de ver los resultados.
- Los pesos de la combinación de los diez no se interpretan uno por uno: la luminosidad, la escala de grises y la primera componente miden casi lo mismo (el brillo) y se reparten el peso de forma inestable.

**Cómo queda la etapa 2.** Mapa principal: tercera componente principal por imagen, signo por oscuridad local. Referencia clásica: verde. Mapas complementarios para la combinación de la etapa 5: segunda componente, verde, azul y escala de grises. La combinación de mapas ya mejora a la tercera componente sola con los mapas crudos, que es lo que pidió la devolución: aprovechar el color con varios mapas.
